# MedFind: Error-Tolerant Drug Lookup System

## Hybrid Retrieval System using Phonetic Encoding + Machine Learning

**Author:** Adun Gorokgodage (20221463)

**Supervised by:** Mr. Cassim Farook

**Date:** February 2026

---

### System Architecture:
1. **Layer 1 - Candidate Generation:** Phonetic encoding (Soundex, Metaphone, Double Metaphone)
2. **Layer 2 - ML Re-ranking:** LightGBM learning-to-rank model
3. **Layer 3 - Hybrid Ensemble:** Combines edit distance + ML predictions


## 1. Data Loading and Preprocessing

Load DrugBank vocabulary and prepare data for phonetic matching.

In [1]:
import os

# Change to the MedFind directory
os.chdir(r'C:\Users\User\OneDrive - University of Westminster\Medfind')

# Verify
print(f"Working directory: {os.getcwd()}")
print(f"Data folder exists: {os.path.exists('Data')}")

Working directory: C:\Users\User\OneDrive - University of Westminster\Medfind
Data folder exists: True


In [2]:
# Import required libraries
import pandas as pd
from rapidfuzz.distance import Levenshtein, JaroWinkler, Jaro

# Load DrugBank vocabulary
df = pd.read_csv(r"C:\Users\User\OneDrive - University of Westminster\Medfind\Data\drugbank vocabulary.csv")

# Select relevant columns and remove entries without common names
df = df[['DrugBank ID', 'Common name', 'Synonyms']]
df = df.dropna(subset=['Common name'])

print(f"Loaded {len(df)} drug entries")
df.head()

Loaded 19830 drug entries


,DrugBank ID,Common name,Synonyms
0,DB00001,Lepirudin,"[Leu1, Thr2]-63-desulfohirudin | Desulfatohiru..."
1,DB00002,Cetuximab,Cetuximab | Cétuximab | Cetuximabum | Chimeric...
2,DB00003,Dornase alfa,Deoxyribonuclease (human clone 18-1 protein mo...
3,DB00004,Denileukin diftitox,DAB(SUB 389)IL2 | Denileukin | Denileukin dift...
4,DB00005,Etanercept,Etanercept | etanercept-szzs | etanercept-ykro...


In [3]:
import re
import unicodedata

def normalize(text):
    """
    Normalize text for phonetic matching:
    - Convert to lowercase
    - Remove accents and special characters
    - Standardize whitespace
    """
    if pd.isna(text):
        return None
    text = text.lower()
    text = unicodedata.normalize("NFKD", text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Normalize generic drug names
df['generic_norm'] = df['Common name'].apply(normalize)
print(f"Normalized {len(df)} drug names")

Normalized 19830 drug names


In [4]:
def split_synonyms(s):
    """
    Split synonym strings and normalize each synonym.
    Synonyms are separated by | or ; in the dataset.
    """
    if pd.isna(s):
        return []
    return [normalize(x) for x in re.split(r'[|;]', s) if x.strip()]

# Process synonyms for each drug
df['syn_list'] = df['Synonyms'].apply(split_synonyms)
df[['Common name', 'Synonyms', 'syn_list']].head()

,Common name,Synonyms,syn_list
0,Lepirudin,"[Leu1, Thr2]-63-desulfohirudin | Desulfatohiru...","[leu thr desulfohirudin, desulfatohirudin, hir..."
1,Cetuximab,Cetuximab | Cétuximab | Cetuximabum | Chimeric...,"[cetuximab, ce tuximab, cetuximabum, chimeric ..."
2,Dornase alfa,Deoxyribonuclease (human clone 18-1 protein mo...,"[deoxyribonuclease human clone protein moiety,..."
3,Denileukin diftitox,DAB(SUB 389)IL2 | Denileukin | Denileukin dift...,"[dab sub il, denileukin, denileukin diftitox, ..."
4,Etanercept,Etanercept | etanercept-szzs | etanercept-ykro...,"[etanercept, etanercept szzs, etanercept ykro,..."


In [5]:
# Build lookup table with both generic names and synonyms
records = []

for _, row in df.iterrows():
    # Add generic name
    records.append({
        'drugbank_id': row['DrugBank ID'],
        'canonical': row['generic_norm'],
        'term': row['generic_norm'],
        'source': 'generic'
    })
    
    # Add all synonyms
    for syn in row['syn_list']:
        records.append({
            'drugbank_id': row['DrugBank ID'],
            'canonical': row['generic_norm'],
            'term': syn,
            'source': 'synonym'
        })

lookup_df = pd.DataFrame(records).dropna()
print(f"Created lookup table with {len(lookup_df)} entries")
print(f"  - Generic names: {len(lookup_df[lookup_df['source']=='generic'])}")
print(f"  - Synonyms: {len(lookup_df[lookup_df['source']=='synonym'])}")
lookup_df.head()

Created lookup table with 71885 entries
  - Generic names: 19830
  - Synonyms: 52055


,drugbank_id,canonical,term,source
0,DB00001,lepirudin,lepirudin,generic
1,DB00001,lepirudin,leu thr desulfohirudin,synonym
2,DB00001,lepirudin,desulfatohirudin,synonym
3,DB00001,lepirudin,hirudin variant,synonym
4,DB00001,lepirudin,lepirudin,synonym


In [ ]:
# ========================================
# ADD PHONETIC CODES TO LOOKUP_DF
# ========================================

import jellyfish

print("Adding phonetic codes to lookup database...")

# Apply phonetic encoding algorithms to all terms
lookup_df['soundex'] = lookup_df['term'].apply(jellyfish.soundex)
lookup_df['metaphone'] = lookup_df['term'].apply(jellyfish.metaphone)

# NYSIIS for expanded phonetic matching
lookup_df['nysiis'] = lookup_df['term'].apply(jellyfish.nysiis)

# Double Metaphone not available in jellyfish - use metaphone as fallback
lookup_df['dmetaphone'] = lookup_df['metaphone']  # Same as metaphone

print("\n✓ Phonetic encodings applied:")
print(f"  - Soundex: {lookup_df['soundex'].nunique()} unique codes")
print(f"  - Metaphone: {lookup_df['metaphone'].nunique()} unique codes")
print(f"  - NYSIIS: {lookup_df['nysiis'].nunique()} unique codes")
print(f"  - Double Metaphone (fallback): {lookup_df['dmetaphone'].nunique()} unique codes")

# Preview
print("\nSample of phonetic codes:")
lookup_df[['term', 'soundex', 'metaphone', 'nysiis']].head(10)

In [7]:
# ========================================
# PRE-COMPUTE ALL PHONETIC CODES
# ========================================

print("Pre-computing phonetic codes for all drugs...")

import jellyfish

# Add NYSIIS codes
if 'nysiis' not in lookup_df.columns:
    lookup_df['nysiis'] = lookup_df['term'].apply(lambda x: jellyfish.nysiis(x) if x else '')
    print(f"✓ Added NYSIIS codes")

# Verify we have all phonetic codes
print(f"\nPhonetic codes available:")
print(f"  - Soundex: {lookup_df['soundex'].nunique()} unique codes")
print(f"  - Metaphone: {lookup_df['metaphone'].nunique()} unique codes")
print(f"  - NYSIIS: {lookup_df['nysiis'].nunique()} unique codes")

print(f"\nTotal drugs: {len(lookup_df)}")

Pre-computing phonetic codes for all drugs...

Phonetic codes available:
  - Soundex: 4464 unique codes
  - Metaphone: 45480 unique codes
  - NYSIIS: 46303 unique codes

Total drugs: 71885


## 2. Phonetic Encoding

Apply Soundex, Metaphone, and Double Metaphone algorithms for phonetic matching.

In [8]:
import jellyfish

# Apply phonetic encoding algorithms to all terms
lookup_df['soundex'] = lookup_df['term'].apply(jellyfish.soundex)
lookup_df['metaphone'] = lookup_df['term'].apply(jellyfish.metaphone)

# Double Metaphone not available in this jellyfish version - use metaphone as fallback
print("⚠ Using Metaphone (single) as Double Metaphone not available in jellyfish 1.2.1")
lookup_df['dmetaphone'] = lookup_df['metaphone']  # Use same as metaphone

print("\nPhonetic encodings applied:")
print(f"  Soundex: {lookup_df['soundex'].nunique()} unique codes")
print(f"  Metaphone: {lookup_df['metaphone'].nunique()} unique codes")
print(f"  Double Metaphone (fallback): {lookup_df['dmetaphone'].nunique()} unique codes")
lookup_df[['term', 'soundex', 'metaphone', 'dmetaphone']].head(10)

⚠ Using Metaphone (single) as Double Metaphone not available in jellyfish 1.2.1

Phonetic encodings applied:
  Soundex: 4464 unique codes
  Metaphone: 45480 unique codes
  Double Metaphone (fallback): 45480 unique codes


,term,soundex,metaphone,dmetaphone
0,lepirudin,L163,LPRTN,LPRTN
1,leu thr desulfohirudin,L363,L 0R TSLFHRTN,L 0R TSLFHRTN
2,desulfatohirudin,D241,TSLFTHRTN,TSLFTHRTN
3,hirudin variant,H635,HRTN FRNT,HRTN FRNT
4,lepirudin,L163,LPRTN,LPRTN
5,lepirudin recombinant,L163,LPRTN RKMBNNT,LPRTN RKMBNNT
6,r hirudin,R635,R HRTN,R HRTN
7,cetuximab,C325,STKSMB,STKSMB
8,cetuximab,C325,STKSMB,STKSMB
9,ce tuximab,C325,S TKSMB,S TKSMB


## 3. Synthetic Data Generation

Generate realistic misspellings for training:
- **Phonetic errors:** Simulate pronunciation-based misspellings
- **Typo errors:** Simulate keyboard/visual errors

In [9]:
import random

def generate_misspelling(word):
    """
    Generate random typo errors:
    - delete: Remove a character
    - swap: Transpose adjacent characters
    - replace: Substitute with random character
    - insert: Add random character
    """
    if not isinstance(word, str) or len(word) < 4:
        return word

    ops = ['delete', 'swap', 'replace', 'insert']
    op = random.choice(ops)
    i = random.randint(1, len(word)-2)

    if op == 'delete':
        return word[:i] + word[i+1:]
    if op == 'swap':
        return word[:i] + word[i+1] + word[i] + word[i+2:]
    if op == 'replace':
        return word[:i] + random.choice('abcdefghijklmnopqrstuvwxyz') + word[i+1:]
    if op == 'insert':
        return word[:i] + random.choice('abcdefghijklmnopqrstuvwxyz') + word[i:]
    return word

# Test the generator
examples = ['amoxicillin', 'ibuprofen', 'paracetamol']
print("Example typo errors:")
for word in examples:
    print(f"  {word:15s} → {generate_misspelling(word)}")

Example typo errors:
  amoxicillin     → amoxiclilin
  ibuprofen       → ibuprofn
  paracetamol     → paracetmaol


In [10]:
def generate_phonetic_misspelling(word):
    """Generate realistic phonetic misspellings"""
    if not isinstance(word, str) or len(word) < 4:
        return word
    
    import random
    
    # Common phonetic confusions
    phonetic_swaps = {
        'ph': 'f',
        'f': 'ph',
        'c': 'k',
        'k': 'c',
        's': 'z',
        'z': 's',
        'x': 'ks',
        'tion': 'shun',
        'ai': 'ay',
        'ei': 'ee',
        'ou': 'ow',
        'ough': 'off',
    }
    
    word_lower = word.lower()
    
    # Try phonetic substitution first
    for original, replacement in phonetic_swaps.items():
        if original in word_lower:
            if random.random() < 0.6:
                return word_lower.replace(original, replacement, 1)
    
    # Vowel confusion
    vowels = 'aeiou'
    word_list = list(word_lower)
    vowel_positions = [i for i, c in enumerate(word_list[1:-1], start=1) if c in vowels]
    
    if vowel_positions and random.random() < 0.7:
        pos = random.choice(vowel_positions)
        current_vowel = word_list[pos]
        similar_vowels = {
            'a': ['e', 'o'],
            'e': ['a', 'i'],
            'i': ['e', 'y'],
            'o': ['a', 'u'],
            'u': ['o'],
        }
        new_vowel = random.choice(similar_vowels.get(current_vowel, vowels))
        word_list[pos] = new_vowel
        return ''.join(word_list)
    
    # Double consonants
    if random.random() < 0.3:
        consonants = 'bcdfghjklmnpqrstvwxz'
        consonant_positions = [i for i, c in enumerate(word_list[1:-1], start=1) if c in consonants]
        if consonant_positions:
            pos = random.choice(consonant_positions)
            if pos + 1 < len(word_list) and word_list[pos] == word_list[pos + 1]:
                word_list.pop(pos)
            else:
                word_list.insert(pos + 1, word_list[pos])
            return ''.join(word_list)
    
    # Fallback to random error
    return generate_misspelling(word)

## 4. Machine Learning Model Training

### LightGBM Learning-to-Rank Model

**Features:**
- Levenshtein distance (edit distance)
- Jaro-Winkler similarity
- Length ratio
- Prefix match
- Phonetic matches (Soundex, Metaphone, Double Metaphone)

**Objective:** Rank candidates by relevance to the query

In [11]:
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import GroupShuffleSplit
from collections import Counter

In [ ]:
# ────────────────────────────────────────────────────────────────
#  B) Feature engineering function
# ────────────────────────────────────────────────────────────────

def extract_features(query_norm, term_norm, source):
    """
    Returns dict of features for one (query, candidate) pair
    NOW WITH EXPANDED PHONETIC FEATURES (not used for filtering!)
    """
    if not term_norm or not query_norm:
        return {
            'lev_dist'        : 999,
            'lev_norm'        : 1.0,
            'jw_sim'          : 0.0,
            'jaro_sim'        : 0.0,
            'len_ratio'       : 1.0,
            'is_generic'      : 0.0,
            'prefix_match'    : 0.0,
            'soundex_match'   : 0.0,
            'metaphone_match' : 0.0,
            'nysiis_match'    : 0.0,
            'match_rating'    : 0.0,
            'phonetic_score'  : 0.0
        }
    
    lev_d = Levenshtein.distance(query_norm, term_norm)
    max_len = max(len(query_norm), len(term_norm))
    
    # Calculate all phonetic matches
    soundex_match = 1.0 if jellyfish.soundex(query_norm) == jellyfish.soundex(term_norm) else 0.0
    metaphone_match = 1.0 if jellyfish.metaphone(query_norm) == jellyfish.metaphone(term_norm) else 0.0
    nysiis_match = 1.0 if jellyfish.nysiis(query_norm) == jellyfish.nysiis(term_norm) else 0.0
    
    # Match Rating Comparison returns True/False/None
    try:
        match_rating_result = jellyfish.match_rating_comparison(query_norm, term_norm)
        match_rating = 1.0 if match_rating_result else 0.0
    except:
        match_rating = 0.0
    
    # Combined phonetic score (average of all 4 methods)
    phonetic_score = (soundex_match + metaphone_match + nysiis_match + match_rating) / 4.0
    
    return {
        'lev_dist'        : lev_d,
        'lev_norm'        : lev_d / max_len if max_len > 0 else 0.0,
        'jw_sim'          : JaroWinkler.similarity(query_norm, term_norm),
        'jaro_sim'        : Jaro.similarity(query_norm, term_norm),
        'len_ratio'       : len(term_norm) / len(query_norm) if len(query_norm) > 0 else 1.0,
        'is_generic'      : 1.0 if source == 'generic' else 0.0,
        'prefix_match'    : 1.0 if term_norm.startswith(query_norm[:3]) else 0.0,
        
        # Individual phonetic features
        'soundex_match'   : soundex_match,
        'metaphone_match' : metaphone_match,
        'nysiis_match'    : nysiis_match,
        'match_rating'    : match_rating,
        
        # Combined phonetic score
        'phonetic_score'  : phonetic_score
    }

In [ ]:
def train_ranker(X, y, group):
    #Train LightGBM ranking model WITH FEATURE NAMES
    
    # CRITICAL: Define feature names in EXACT order
    FEATURE_NAMES = [
        'lev_dist',
        'lev_norm',
        'jw_sim',
        'jaro_sim',
        'len_ratio',
        'is_generic',
        'prefix_match',
        'soundex_match',
        'metaphone_match',
        'nysiis_match',
        'match_rating',
        'phonetic_score'
    ]
    
    if group.sum() != len(X):
        raise ValueError(f"Group sizes inconsistent: sum={group.sum()}, rows={len(X)}")
    
    # Create dataset WITH feature names
    train_data = lgb.Dataset(
        X, 
        label=y, 
        group=group,
        feature_name=FEATURE_NAMES
    )
    
    params = {
        'objective': 'lambdarank',
        'metric': 'ndcg',
        'ndcg_eval_at': [1, 5],
        'learning_rate': 0.05,
        'num_leaves': 31,
        'min_data_in_leaf': 5,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'verbose': -1,
        'num_iterations': 150
    }
    
    print(f"\n Training LightGBM with {len(FEATURE_NAMES)} features")
    print(f"   Features: {FEATURE_NAMES}")
    
    ranker = lgb.train(
        params,
        train_data,
        num_boost_round=150,
        valid_sets=[train_data],
        valid_names=['training']
    )
    
    print(f"\n✓ Model trained successfully")
    print(f"  Feature names: {ranker.feature_name()}")
    
    return ranker


In [ ]:
# ────────────────────────────────────────────────────────────────
#  E) Inference: re-rank candidates with the trained model
# ────────────────────────────────────────────────────────────────

def rank_candidates(query, lookup_df, model, top_k=5):
    """
    Rank candidates WITHOUT phonetic filtering
    Searches all drugs, lets ML model use phonetic features for scoring
    """
    import pandas as pd
    import numpy as np
    from rapidfuzz.distance import Levenshtein
    
    q_norm = normalize(query)
    
    if not q_norm:
        # Return empty DataFrame if query is invalid
        return pd.DataFrame(columns=['canonical', 'term', 'rank_score'])
    
    print(f"Searching all {len(lookup_df)} drugs (no phonetic filter)...")
    
    # ========================================
    # Search ALL drugs
    # ========================================
    
    # Use entire database as candidates
    candidates = lookup_df.copy()
    
    # ========================================
    # EXTRACT FEATURES FOR ALL CANDIDATES
    # ========================================
    
    feature_list = []
    for _, row in candidates.iterrows():
        feats = extract_features(q_norm, row['term'], row['source'])
        feature_list.append(feats)
    
    if not feature_list:
        # Return empty DataFrame if no features extracted
        return pd.DataFrame(columns=['canonical', 'term', 'rank_score'])
    
    # ========================================
    # CONVERT TO ARRAY FOR ML MODEL
    # ========================================
    
    feature_names = list(feature_list[0].keys())
    X_test = [[feat_dict[fn] for fn in feature_names] for feat_dict in feature_list]
    X_test = np.array(X_test, dtype=np.float32)
    
    # ========================================
    # PREDICT SCORES WITH ML MODEL
    # ========================================
    
    scores = model.predict(X_test)
    
    # ========================================
    # COMBINE RESULTS
    # ========================================
    
    candidates = candidates.copy()
    candidates['rank_score'] = scores
    
    # Sort by score (descending)
    candidates = candidates.sort_values('rank_score', ascending=False)
    
    # Return top K
    result = candidates.head(top_k)[['canonical', 'term', 'rank_score']].reset_index(drop=True)
    
    print(f"✓ Found {len(result)} results")
    
    return result

## 5. Model Training

Generate training data with mixed phonetic + typo errors and train the LightGBM ranker.

In [ ]:
# ────────────────────────────────────────────────────────────────
#  FUNCTION: BUILD TRAINING DATA
# ────────────────────────────────────────────────────────────────

def build_training_data(lookup_df, sample_queries_with_labels, group_size_limit=200):
    
    #Searches ALL drugs by edit distance instead
    
    import numpy as np
    from rapidfuzz.distance import Levenshtein
    
    X, y, group_sizes, queries = [], [], [], []
    
    print(f"Processing {len(sample_queries_with_labels)} training queries...")
    
    for idx, (raw_query, true_canonical) in enumerate(sample_queries_with_labels):
        if (idx + 1) % 500 == 0:
            print(f"  Processed {idx + 1}/{len(sample_queries_with_labels)} queries...")
        
        q_norm = normalize(raw_query)
        if not q_norm:
            continue

        
        lookup_df['temp_lev'] = lookup_df['term'].apply(
            lambda t: Levenshtein.distance(q_norm, t) if t else 999
        )
        
        # Take top N candidates by edit distance
        candidates = lookup_df.nsmallest(group_size_limit, 'temp_lev').copy()
        lookup_df.drop(columns=['temp_lev'], inplace=True)

        # ========================================
        # EXTRACT FEATURES FOR EACH CANDIDATE
        # ========================================
        
        group_data = []
        for _, row in candidates.iterrows():
            feats = extract_features(q_norm, row['term'], row['source'])
            group_data.append(feats)

        # ========================================
        # CREATE RELEVANCE LABELS
        # ========================================
        
        # Find if true answer is in this group
        true_idx = -1
        for i, row in enumerate(candidates.itertuples()):
            if row.canonical == true_canonical:
                true_idx = i
                break

        # Assign labels: 1.0 for correct, 0.0 for others
        labels = [0.0] * len(group_data)
        if true_idx >= 0:
            labels[true_idx] = 1.0

        # ========================================
        # APPEND TO TRAINING DATA
        # ========================================
        
        X.extend(group_data)
        y.extend(labels)
        group_sizes.append(len(group_data))
        queries.append(raw_query)

    # ========================================
    # CONVERT TO ARRAYS
    # ========================================
    
    if not X:
        print("❌ ERROR: No training data generated!")
        return None, None, None, None
    
    # Convert feature dicts to lists
    feature_names = list(X[0].keys())
    X_array = [[feat_dict[fn] for fn in feature_names] for feat_dict in X]
    
    # Convert to numpy arrays
    X_array = np.array(X_array, dtype=np.float32)
    y_array = np.array(y, dtype=np.float32)
    group_array = np.array(group_sizes, dtype=np.int32)
    
    print(f"\n✓ Training data built successfully:")
    print(f"  - Total examples: {len(X_array)}")
    print(f"  - Total queries: {len(queries)}")
    print(f"  - Features per example: {len(feature_names)}")
    print(f"  - Average candidates per query: {len(X_array) / len(queries):.1f}")
    print(f"  - Feature names: {feature_names}")
    
    # ========================================
    # RETURN THE DATA
    # ========================================
    
    return X_array, y_array, group_array, queries

In [ ]:
# ────────────────────────────────────────────────────────────────
#  TRAINING DATA GENERATION
# ────────────────────────────────────────────────────────────────
import random
random.seed(42)

# Get unique generic drugs
generic_df = lookup_df[lookup_df['source'] == 'generic'][['canonical']].drop_duplicates()

print(f"Generating training pairs from {len(generic_df)} unique drugs...")

# Generate mixed phonetic + typo errors
example_pairs = []
error_types = {'phonetic': 0, 'typo': 0, 'unchanged': 0}

for _, row in generic_df.iterrows():
    canonical = row['canonical']
    if not canonical or len(canonical) < 4:
        continue
    
    # Generate 5 misspellings per drug
    for i in range(5):
        if i < 2:  # First 2: phonetic errors (40%)
            misspelled = generate_phonetic_misspelling(canonical)
            error_type = 'phonetic'
        else:      # Last 3: typo errors (60%)
            misspelled = generate_misspelling(canonical)
            error_type = 'typo'
        
        if misspelled != canonical:
            example_pairs.append((misspelled, canonical))
            error_types[error_type] += 1
        else:
            error_types['unchanged'] += 1

print(f"\nGenerated {len(example_pairs)} training pairs:")
print(f"  - Phonetic errors: {error_types['phonetic']} ({error_types['phonetic']/len(example_pairs)*100:.1f}%)")
print(f"  - Typo errors: {error_types['typo']} ({error_types['typo']/len(example_pairs)*100:.1f}%)")
print(f"  - Unchanged: {error_types['unchanged']}")

# Save full dataset
pd.DataFrame(example_pairs, columns=['misspelled', 'canonical']).to_csv(
    r'Data\synthetic_training_data_mixed.csv', 
    index=False
)
print(f"\nSaved full dataset to synthetic_training_data_mixed.csv")

# Sample down for training (to keep training time reasonable)
example_pairs_sampled = random.sample(example_pairs, min(3000, len(example_pairs)))
print(f"Using {len(example_pairs_sampled)} pairs for training")

# Show some examples
print("\n" + "="*60)
print("SAMPLE PHONETIC ERRORS:")
print("="*60)
phonetic_samples = [(m, c) for m, c in example_pairs if generate_phonetic_misspelling(c) != c][:5]
for misspelled, canonical in phonetic_samples[:5]:
    print(f"  {canonical:20s} → {misspelled}")

print("\n" + "="*60)
print("SAMPLE TYPO ERRORS:")
print("="*60)
typo_samples = [(m, c) for m, c in example_pairs if m != c][:5]
for misspelled, canonical in typo_samples[:5]:
    print(f"  {canonical:20s} → {misspelled}")

# ────────────────────────────────────────────────────────────────
#  BUILD TRAINING DATASET
# ────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("BUILDING TRAINING DATASET...")
print("="*60)

X_train, y_train, group_train, _ = build_training_data(
    lookup_df, 
    example_pairs_sampled, 
    group_size_limit=40
)


# ────────────────────────────────────────────────────────────────
#  TRAIN MODEL
# ────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("TRAINING MODEL...")
print("="*60)

ranker = train_ranker(X_train, y_train, group_train)

# ────────────────────────────────────────────────────────────────
#  TEST ON SAMPLE QUERIES
# ────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("TESTING ON SAMPLE QUERIES")
print("="*60)

test_queries = [
    ("amoxcillin", "amoxicillin"),      # Typo
    ("ibuprophen", "ibuprofen"),        # Phonetic (f→ph)
    ("paracetomol", "paracetamol"),     # Phonetic (a→o)
    ("asprin", "aspirin"),              # Typo (missing 'i')
    ("omeprazzol", "omeprazole"),       # Phonetic (double z)
]

for query, expected in test_queries:
    print(f"\n--- Query: '{query}' (Expected: {expected}) ---")
    ranked = rank_candidates(query, lookup_df, ranker, top_k=3)
    if not ranked.empty:
        top1 = ranked.iloc[0]['canonical']
        correct = "✓" if top1 == expected else "✗"
        print(f"{correct} Top result: {top1}")
        print(ranked[['canonical', 'rank_score']].to_string(index=False))
    else:
        print("✗ No results found")



Generating training pairs from 18546 unique drugs...

Generated 87384 training pairs:
  - Phonetic errors: 35151 (40.2%)
  - Typo errors: 52233 (59.8%)
  - Unchanged: 571

Saved full dataset to synthetic_training_data_mixed.csv
Using 3000 pairs for training

SAMPLE PHONETIC ERRORS:
  lepirudin            → lipirudin
  lepirudin            → leperudin
  lepirudin            → lepirdin
  lepirudin            → laepirudin
  lepirudin            → leirudin

SAMPLE TYPO ERRORS:
  lepirudin            → lipirudin
  lepirudin            → leperudin
  lepirudin            → lepirdin
  lepirudin            → laepirudin
  lepirudin            → leirudin

BUILDING TRAINING DATASET...
Processing 3000 training queries...
  Processed 500/3000 queries...
  Processed 1000/3000 queries...
  Processed 1500/3000 queries...
  Processed 2000/3000 queries...
  Processed 2500/3000 queries...
  Processed 3000/3000 queries...

✓ Training data built successfully:
  - Total examples: 120000
  - Total queries: 30

## 6. Model Evaluation

Test the system on realistic misspellings and calculate performance metrics.

In [17]:
test_misspellings = [
    ("amoxcillin", "amoxicillin"),      # missing 'i'
    ("asprin", "acetylsalicylic acid"), # missing 'i'
    ("paracetomol", "acetaminophen"),   # 'o' instead of 'a'
    ("ibuprophen", "ibuprofen"),        # 'ph' instead of 'f'
    ("omeprazzol", "omeprazole"),       # double 'z'
]

for misspelled, expected in test_misspellings:
    result = rank_candidates(misspelled, lookup_df, ranker, top_k=1)
    print(f"{misspelled} → {result.iloc[0]['canonical']} (expected: {expected})")

Searching all 71885 drugs (no phonetic filter)...
✓ Found 1 results
amoxcillin → amoxicillin (expected: amoxicillin)
Searching all 71885 drugs (no phonetic filter)...
✓ Found 1 results
asprin → acetylsalicylic acid (expected: acetylsalicylic acid)
Searching all 71885 drugs (no phonetic filter)...
✓ Found 1 results
paracetomol → acetaminophen (expected: acetaminophen)
Searching all 71885 drugs (no phonetic filter)...
✓ Found 1 results
ibuprophen → ibuprofen (expected: ibuprofen)
Searching all 71885 drugs (no phonetic filter)...
✓ Found 1 results
omeprazzol → omeprazole (expected: omeprazole)


In [18]:
def evaluate_model(test_pairs, lookup_df, ranker):
    correct_top1 = 0
    correct_top5 = 0
    
    for query, true_canonical in test_pairs:
        results = rank_candidates(query, lookup_df, ranker, top_k=5)
        if results.empty:
            continue
            
        top_canonicals = results['canonical'].tolist()
        if top_canonicals[0] == true_canonical:
            correct_top1 += 1
            correct_top5 += 1
        elif true_canonical in top_canonicals:
            correct_top5 += 1
    
    acc_top1 = correct_top1 / len(test_pairs)
    acc_top5 = correct_top5 / len(test_pairs)
    
    return acc_top1, acc_top5

In [ ]:
# ────────────────────────────────────────────────────────────────
#  OPTIMIZED EVALUATION CELL
# ────────────────────────────────────────────────────────────────

import random
from sklearn.metrics import confusion_matrix
import pickle
import numpy as np

def rank_candidates_fast(query, lookup_df, model, feature_cache, top_k=5):
    """
    FAST version: Uses pre-computed features when possible
    """
    q_norm = normalize(query)
    
    if not q_norm:
        return pd.DataFrame(columns=['canonical', 'term', 'rank_score'])
    
    # Check if we've already computed features for this query
    cache_key = q_norm
    if cache_key in feature_cache:
        X_test = feature_cache[cache_key]
    else:
        # Extract features (this is the slow part)
        feature_list = []
        for _, row in lookup_df.iterrows():
            feats = extract_features(q_norm, row['term'], row['source'])
            feature_list.append(feats)
        
        feature_names = list(feature_list[0].keys())
        X_test = [[feat_dict[fn] for fn in feature_names] for feat_dict in feature_list]
        X_test = np.array(X_test, dtype=np.float32)
        
        # Cache for reuse
        feature_cache[cache_key] = X_test
    
    # Predict scores
    scores = model.predict(X_test)
    
    # Combine results
    candidates = lookup_df.copy()
    candidates['rank_score'] = scores
    candidates = candidates.sort_values('rank_score', ascending=False)
    
    return candidates.head(top_k)[['canonical', 'term', 'rank_score']].reset_index(drop=True)


def evaluate_model_fast(test_pairs, lookup_df, ranker, top_k=5):
    """Evaluate model with progress updates"""
    correct_top1 = 0
    correct_top5 = 0
    total_queries = 0
    no_results = 0
    y_true = []
    y_pred = []
    
    # Feature cache to avoid recomputing
    feature_cache = {}
    
    print(f"\nEvaluating {len(test_pairs)} queries...")
    
    for idx, (query, true_canonical) in enumerate(test_pairs):
        # Progress update every 100 queries
        if (idx + 1) % 100 == 0:
            print(f"  Processed {idx + 1}/{len(test_pairs)} queries... "
                  f"(Accuracy so far: {correct_top1/total_queries*100:.1f}%)")
        
        total_queries += 1
        results = rank_candidates_fast(query, lookup_df, ranker, feature_cache, top_k=top_k)
        
        if results.empty:
            no_results += 1
            y_true.append(1)
            y_pred.append(0)
            continue
        
        top_canonicals = results['canonical'].tolist()
        
        if top_canonicals[0] == true_canonical:
            correct_top1 += 1
            correct_top5 += 1
            y_true.append(1)
            y_pred.append(1)
        elif true_canonical in top_canonicals:
            correct_top5 += 1
            y_true.append(1)
            y_pred.append(0)
        else:
            y_true.append(1)
            y_pred.append(0)
    
    acc_top1 = correct_top1 / total_queries
    acc_top5 = correct_top5 / total_queries
    recall = correct_top5 / total_queries
    precision = correct_top1 / (total_queries - no_results) if total_queries > no_results else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    
    return {
        'total_queries': total_queries,
        'top_1_accuracy': acc_top1,
        'top_5_accuracy': acc_top5,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'no_results': no_results,
        'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn,
        'fpr': fpr
    }


def generate_test_set(lookup_df, n_samples=1000, error_rate=0.5):
    """Generate test set with misspellings"""
    unique_drugs = lookup_df[lookup_df['source'] == 'generic']['canonical'].unique()
    sampled_drugs = random.sample(list(unique_drugs), min(n_samples, len(unique_drugs)))
    
    test_pairs = []
    for drug in sampled_drugs:
        if random.random() < error_rate:
            misspelled = generate_misspelling(drug)
            if misspelled != drug:
                test_pairs.append((misspelled, drug))
            else:
                test_pairs.append((drug, drug))
        else:
            test_pairs.append((drug, drug))
    
    return test_pairs


# ────────────────────────────
#  TEST SET
# ────────────────────────────

print("="*60)
print("GENERATING TEST SET")
print("="*60)

random.seed(123)

# REDUCED from 1000 to 300 for faster evaluation
test_pairs = generate_test_set(lookup_df, n_samples=300, error_rate=0.5)

print(f"✓ Generated {len(test_pairs)} test pairs (50% misspelled)")


# ────────────────────────────────────────────────────────────────
#  EVALUATE MODEL
# ────────────────────────────────────────────────────────────────

print("\n" + "="*60)
print("EVALUATING MODEL")
print("="*60)

import time
start_time = time.time()

metrics = evaluate_model_fast(test_pairs, lookup_df, ranker, top_k=5)

elapsed_time = time.time() - start_time

print(f"\n✓ Evaluation completed in {elapsed_time:.1f} seconds ({elapsed_time/60:.1f} minutes)")
print(f"  Average time per query: {elapsed_time/len(test_pairs):.2f} seconds")


# ────────────────────────────────────────────────────────────────
#  PRINT REPORT
# ────────────────────────────────────────────────────────────────

print("\n" + "="*60)
print("MEDFIND EVALUATION REPORT")
print("="*60)
print(f"\nTotal Test Queries: {metrics['total_queries']}")
print(f"Queries with No Results: {metrics['no_results']}")

print("\n" + "-"*60)
print("ACCURACY METRICS")
print("-"*60)
print(f"Top-1 Accuracy: {metrics['top_1_accuracy']:.3f} ({metrics['top_1_accuracy']*100:.1f}%)")
print(f"Top-5 Accuracy: {metrics['top_5_accuracy']:.3f} ({metrics['top_5_accuracy']*100:.1f}%)")
print(f"\nPrecision: {metrics['precision']:.3f}")
print(f"Recall: {metrics['recall']:.3f}")
print(f"F1-Score: {metrics['f1_score']:.3f}")

print("\n" + "-"*60)
print("CONFUSION MATRIX (Top-1 Classification)")
print("-"*60)
print(f"True Positives (Correct): {metrics['tp']}")
print(f"True Negatives: {metrics['tn']}")
print(f"False Positives: {metrics['fp']}")
print(f"False Negatives (Missed): {metrics['fn']}")
print(f"False Positive Rate: {metrics['fpr']:.3f}")

print("\n" + "-"*60)
print("NFR01 COMPLIANCE CHECK (95% Accuracy Target)")
print("-"*60)
if metrics['top_1_accuracy'] >= 0.95:
    print(f"✓ PASSED: {metrics['top_1_accuracy']*100:.1f}% >= 95%")
else:
    print(f"✗ FAILED: {metrics['top_1_accuracy']*100:.1f}% < 95%")
    print(f"  Gap: {(0.95 - metrics['top_1_accuracy'])*100:.1f}% needed")

print("\n" + "="*60)


# ────────────────────────────────────────────────────────────────
#  ERROR ANALYSIS 
# ────────────────────────────────────────────────────────────────

print("\nAnalyzing failed queries (sampling 50 for speed)...")

feature_cache = {}  # Reuse cache
failed_queries = []

for query, true_canonical in test_pairs[:50]:  # Only first 50 for speed
    results = rank_candidates_fast(query, lookup_df, ranker, feature_cache, top_k=5)
    if results.empty or results.iloc[0]['canonical'] != true_canonical:
        failed_queries.append({
            'query': query,
            'expected': true_canonical,
            'got': results.iloc[0]['canonical'] if not results.empty else 'NO RESULTS',
            'top_5': results['canonical'].tolist() if not results.empty else []
        })

if failed_queries:
    pd.DataFrame(failed_queries).to_csv('error_analysis_sample.csv', index=False)
    print(f"✓ Analyzed {len(failed_queries)} failures (saved to error_analysis_sample.csv)")


# ────────────────────────────────────────────────────────────────
#  SAVE RESULTS
# ────────────────────────────────────────────────────────────────

print("\nSaving results...")

# Save metrics
results_df = pd.DataFrame([metrics])
results_df.to_csv(r'Data\evaluation_results.csv', index=False)
print("✓ Metrics saved to evaluation_results.csv")

# Save model
import os
os.makedirs(r'Code\Models', exist_ok=True)

with open(r'Code\Models\drug_ranker_model.pkl', 'wb') as f:
    pickle.dump(ranker, f)
print("✓ Model saved to Code/Models/drug_ranker_model.pkl")

# Save lookup_df
lookup_df.to_pickle(r'Code\Models\lookup_df.pkl')
print("✓ Lookup database saved to Code/Models/lookup_df.pkl")

print("\n" + "="*60)
print("✅ EVALUATION COMPLETE!")
print("="*60)

GENERATING TEST SET
✓ Generated 300 test pairs (50% misspelled)

EVALUATING MODEL

Evaluating 300 queries...
  Processed 100/300 queries... (Accuracy so far: 99.0%)
  Processed 200/300 queries... (Accuracy so far: 99.0%)
  Processed 300/300 queries... (Accuracy so far: 99.3%)

✓ Evaluation completed in 749.9 seconds (12.5 minutes)
  Average time per query: 2.50 seconds

MEDFIND EVALUATION REPORT

Total Test Queries: 300
Queries with No Results: 0

------------------------------------------------------------
ACCURACY METRICS
------------------------------------------------------------
Top-1 Accuracy: 0.993 (99.3%)
Top-5 Accuracy: 1.000 (100.0%)

Precision: 0.993
Recall: 1.000
F1-Score: 0.997

------------------------------------------------------------
CONFUSION MATRIX (Top-1 Classification)
------------------------------------------------------------
True Positives (Correct): 298
True Negatives: 0
False Positives: 0
False Negatives (Missed): 2
False Positive Rate: 0.000

--------------

## 7. Hybrid Ensemble System

Combine edit distance (baseline) with ML predictions for improved accuracy.

In [20]:
# ────────────────────────────────────────────────────────────────
#  HELPER FUNCTIONS FOR HYBRID ENSEMBLE
# ────────────────────────────────────────────────────────────────

def get_edit_distance_ranking(query, lookup_df, top_k=5):
    """
    Path 1: Pure edit distance ranking (baseline)
    Returns candidates ranked by Levenshtein distance
    """
    q_norm = normalize(query)
    
    # Phonetic pre-filter for efficiency
    sx = jellyfish.soundex(q_norm)
    mp = jellyfish.metaphone(q_norm)
    
    candidates = lookup_df[
        (lookup_df['soundex'] == sx) | (lookup_df['metaphone'] == mp)
    ].copy()
    
    # Fallback if phonetic finds nothing
    if candidates.empty:
        candidates = lookup_df.copy()
    
    # Rank by edit distance
    candidates['edit_dist'] = candidates['term'].apply(
        lambda t: Levenshtein.distance(q_norm, t)
    )
    
    # Calculate confidence (inverse of distance, normalized)
    max_len = max(len(q_norm), candidates['term'].str.len().max())
    candidates['edit_confidence'] = 1 - (candidates['edit_dist'] / max_len)
    
    result = candidates.sort_values('edit_dist').head(top_k)
    
    return result[['drugbank_id', 'canonical', 'term', 'source', 'edit_dist', 'edit_confidence']]


def get_ml_ranking(query, lookup_df, model, top_k=5):
    """
    Path 2: ML-based ranking (LightGBM)
    Returns candidates ranked by ML model score
    """
    q_norm = normalize(query)
    
    sx = jellyfish.soundex(q_norm)
    mp = jellyfish.metaphone(q_norm)
    
    candidates = lookup_df[
        (lookup_df['soundex'] == sx) | (lookup_df['metaphone'] == mp)
    ].copy()
    
    # Fallback if phonetic finds nothing
    if candidates.empty:
        candidates = lookup_df.copy()
    
    if candidates.empty:
        return pd.DataFrame()
    
    # Compute features
    candidates['lev_dist'] = candidates['term'].apply(lambda t: Levenshtein.distance(q_norm, t))
    candidates['jw_sim'] = candidates['term'].apply(lambda t: JaroWinkler.similarity(q_norm, t))
    
    # Build feature matrix
    X_pred = []
    for _, row in candidates.iterrows():
        feat = extract_features(q_norm, row['term'], row['source'])
        X_pred.append(list(feat.values()))
    
    X_pred = np.array(X_pred)
    
    # Get ML scores
    ml_scores = model.predict(X_pred)
    candidates['ml_score'] = ml_scores
    
    # Convert ML scores to confidence (softmax)
    scores_exp = np.exp(ml_scores - ml_scores.max())
    ml_confidence = (scores_exp / scores_exp.sum()) * 100
    candidates['ml_confidence'] = ml_confidence
    
    result = candidates.sort_values('ml_score', ascending=False).head(top_k)
    
    return result[['drugbank_id', 'canonical', 'term', 'source', 'ml_score', 'ml_confidence']]

In [ ]:
# ────────────────────────────────────────────────────────────────
#  HYBRID ENSEMBLE 
# ────────────────────────────────────────────────────────────────

def hybrid_ensemble_search_v2(query, lookup_df, model, top_k=5, strategy='confidence_weighted'):
    """
    Improved Hybrid Ensemble with better disagreement handling
    
    Strategies:
    - 'confidence_weighted': Use weighted average (default)
    - 'max_confidence': Pick method with highest confidence
    - 'conservative': When disagree, include both in top results
    """
    
    # Get rankings from both methods
    edit_results = get_edit_distance_ranking(query, lookup_df, top_k=10)
    ml_results = get_ml_ranking(query, lookup_df, model, top_k=10)
    
    if edit_results.empty and ml_results.empty:
        return pd.DataFrame()
    
    # Get top-1 from each
    edit_top1 = edit_results.iloc[0]['canonical'] if not edit_results.empty else None
    ml_top1 = ml_results.iloc[0]['canonical'] if not ml_results.empty else None
    
    # Check agreement
    agreement = (edit_top1 == ml_top1) if (edit_top1 and ml_top1) else False
    
    # Merge on canonical name (to avoid duplicates)
    # First get unique canonical from each method
    edit_unique = edit_results.drop_duplicates(subset='canonical')
    ml_unique = ml_results.drop_duplicates(subset='canonical')
    
    merged = pd.merge(
        edit_unique[['canonical', 'edit_dist', 'edit_confidence']],
        ml_unique[['canonical', 'ml_score', 'ml_confidence']],
        on='canonical',
        how='outer'
    )
    
    # Fill missing values
    merged['edit_confidence'] = merged['edit_confidence'].fillna(0)
    merged['ml_confidence'] = merged['ml_confidence'].fillna(0)
    merged['edit_dist'] = merged['edit_dist'].fillna(999)
    merged['ml_score'] = merged['ml_score'].fillna(-999)
    
    # STRATEGY 1: Confidence Weighted (works well when methods agree)
    if strategy == 'confidence_weighted':
        # Normalize ML confidence to 0-1 range
        ml_norm = merged['ml_confidence'] / 100
        
        # Adaptive weighting based on agreement
        if agreement:
            # When agree, trust both equally
            edit_weight = 0.5
            ml_weight = 0.5
            agreement_boost = 0.3  # 30% boost for agreement
        else:
            # When disagree, trust ML more (it's more sophisticated)
            edit_weight = 0.3
            ml_weight = 0.7
            agreement_boost = 0.0
        
        merged['ensemble_score'] = (
            edit_weight * merged['edit_confidence'] + 
            ml_weight * ml_norm
        )
        
        # Boost the agreed-upon result
        if agreement and edit_top1 is not None:
            merged.loc[merged['canonical'] == edit_top1, 'ensemble_score'] += agreement_boost
    
    # STRATEGY 2: Max Confidence (pick the method most confident)
    elif strategy == 'max_confidence':
        merged['ensemble_score'] = merged[['edit_confidence', 'ml_confidence']].max(axis=1)
    
    # STRATEGY 3: Conservative (when disagree, show both prominently)
    elif strategy == 'conservative':
        ml_norm = merged['ml_confidence'] / 100
        merged['ensemble_score'] = (merged['edit_confidence'] + ml_norm) / 2
        
        # If disagreement, boost BOTH top choices
        if not agreement:
            if edit_top1:
                merged.loc[merged['canonical'] == edit_top1, 'ensemble_score'] += 0.15
            if ml_top1:
                merged.loc[merged['canonical'] == ml_top1, 'ensemble_score'] += 0.15
    
    # Add metadata
    merged['agreement'] = agreement
    merged['in_edit_top5'] = merged['canonical'].isin(edit_unique.head(5)['canonical'])
    merged['in_ml_top5'] = merged['canonical'].isin(ml_unique.head(5)['canonical'])
    merged['both_methods'] = merged['in_edit_top5'] & merged['in_ml_top5']
    
    # Sort and deduplicate
    result = merged.sort_values('ensemble_score', ascending=False).drop_duplicates(subset='canonical').head(top_k)
    
    return result


def evaluate_hybrid_system_v2(test_pairs, lookup_df, model, strategy='confidence_weighted'):
    
    #Evaluate the hybrid ensemble system using V2
    
    
    total = len(test_pairs)
    hybrid_correct = 0
    edit_correct = 0
    ml_correct = 0
    agreement_count = 0
    correct_when_agree = 0
    correct_when_disagree = 0
    disagree_count = 0
    
    for query, true_canonical in test_pairs:
        # Hybrid result (V2)
        hybrid_result = hybrid_ensemble_search_v2(query, lookup_df, model, top_k=1, strategy=strategy)
        
        if not hybrid_result.empty:
            hybrid_pred = hybrid_result.iloc[0]['canonical']
            agreed = hybrid_result.iloc[0]['agreement']
            
            if hybrid_pred == true_canonical:
                hybrid_correct += 1
                if agreed:
                    correct_when_agree += 1
                else:
                    correct_when_disagree += 1
            
            if agreed:
                agreement_count += 1
            else:
                disagree_count += 1
        
        # Individual method results (for comparison)
        edit_result = get_edit_distance_ranking(query, lookup_df, top_k=1)
        if not edit_result.empty and edit_result.iloc[0]['canonical'] == true_canonical:
            edit_correct += 1
        
        ml_result = get_ml_ranking(query, lookup_df, model, top_k=1)
        if not ml_result.empty and ml_result.iloc[0]['canonical'] == true_canonical:
            ml_correct += 1
    
    agreement_rate = agreement_count / total
    
    return {
        'total_queries': total,
        'hybrid_accuracy': hybrid_correct / total,
        'edit_accuracy': edit_correct / total,
        'ml_accuracy': ml_correct / total,
        'agreement_rate': agreement_rate,
        'accuracy_when_agree': correct_when_agree / agreement_count if agreement_count > 0 else 0,
        'accuracy_when_disagree': correct_when_disagree / disagree_count if disagree_count > 0 else 0,
        'disagreement_count': disagree_count
    }


def smart_disagreement_resolver(query, lookup_df, model):
    """
    When methods disagree, use heuristics to pick the right answer
    """
    
    edit_results = get_edit_distance_ranking(query, lookup_df, top_k=1)
    ml_results = get_ml_ranking(query, lookup_df, model, top_k=1)
    
    if edit_results.empty or ml_results.empty:
        return ml_results if not ml_results.empty else edit_results
    
    edit_top = edit_results.iloc[0]
    ml_top = ml_results.iloc[0]
    
    # Agreement - easy case
    if edit_top['canonical'] == ml_top['canonical']:
        return pd.DataFrame([{
            'canonical': edit_top['canonical'],
            'confidence': 'HIGH',
            'method': 'both_agree',
            'edit_dist': edit_top['edit_dist'],
            'ml_confidence': ml_top['ml_confidence']
        }])
    
    # Disagreement - apply heuristics
    q_norm = normalize(query)
    
    # Rule 1: Very close edit distance (1-2 chars) → simple typo
    if edit_top['edit_dist'] <= 2:
        return pd.DataFrame([{
            'canonical': edit_top['canonical'],
            'confidence': 'MEDIUM-HIGH',
            'method': 'edit_distance',
            'reason': f'Very close match (distance={edit_top["edit_dist"]})',
            'alternative': ml_top['canonical'],
            'ml_confidence': ml_top['ml_confidence']
        }])
    
    # Rule 2: High ML confidence (>70%)
    if ml_top['ml_confidence'] > 70:
        return pd.DataFrame([{
            'canonical': ml_top['canonical'],
            'confidence': 'MEDIUM-HIGH',
            'method': 'ml_model',
            'reason': f'High ML confidence ({ml_top["ml_confidence"]:.1f}%)',
            'alternative': edit_top['canonical'],
            'edit_dist': edit_top['edit_dist']
        }])
    
    # Rule 3: Can't decide - return both
    return pd.DataFrame([
        {
            'canonical': ml_top['canonical'],
            'confidence': 'LOW',
            'method': 'ml_primary',
            'reason': 'Methods disagree, ML slightly preferred',
            'alternative': edit_top['canonical'],
            'flag': 'MANUAL_REVIEW_RECOMMENDED'
        }
    ])

In [ ]:
def evaluate_hybrid_system_v2(test_pairs, lookup_df, model, strategy='confidence_weighted'):
    """
    Evaluate the hybrid ensemble system using V2
    """
    
    total = len(test_pairs)
    hybrid_correct = 0
    edit_correct = 0
    ml_correct = 0
    agreement_count = 0
    correct_when_agree = 0
    correct_when_disagree = 0
    disagree_count = 0
    
    for query, true_canonical in test_pairs:
        # Hybrid result
        hybrid_result = hybrid_ensemble_search_v2(query, lookup_df, model, top_k=1, strategy=strategy)
        
        if not hybrid_result.empty:
            hybrid_pred = hybrid_result.iloc[0]['canonical']
            agreed = hybrid_result.iloc[0]['agreement']
            
            if hybrid_pred == true_canonical:
                hybrid_correct += 1
                if agreed:
                    correct_when_agree += 1
                else:
                    correct_when_disagree += 1
            
            if agreed:
                agreement_count += 1
            else:
                disagree_count += 1
        
        # Individual method results
        edit_result = get_edit_distance_ranking(query, lookup_df, top_k=1)
        if not edit_result.empty and edit_result.iloc[0]['canonical'] == true_canonical:
            edit_correct += 1
        
        ml_result = get_ml_ranking(query, lookup_df, model, top_k=1)
        if not ml_result.empty and ml_result.iloc[0]['canonical'] == true_canonical:
            ml_correct += 1
    
    agreement_rate = agreement_count / total
    
    return {
        'total_queries': total,
        'hybrid_accuracy': hybrid_correct / total,
        'edit_accuracy': edit_correct / total,
        'ml_accuracy': ml_correct / total,
        'agreement_rate': agreement_rate,
        'accuracy_when_agree': correct_when_agree / agreement_count if agreement_count > 0 else 0,
        'accuracy_when_disagree': correct_when_disagree / disagree_count if disagree_count > 0 else 0,
        'disagreement_count': disagree_count
    }


# Test all three strategies
print("TESTING DIFFERENT ENSEMBLE STRATEGIES")
print("="*60)

for strategy in ['confidence_weighted', 'max_confidence', 'conservative']:
    metrics = evaluate_hybrid_system_v2(test_pairs, lookup_df, ranker, strategy=strategy)
    
    print(f"\n{strategy.upper()}")
    print("-"*60)
    print(f"Hybrid Accuracy:        {metrics['hybrid_accuracy']:.1%}")
    print(f"Edit Distance Only:     {metrics['edit_accuracy']:.1%}")
    print(f"ML Model Only:          {metrics['ml_accuracy']:.1%}")
    print(f"Agreement Rate:         {metrics['agreement_rate']:.1%}")
    print(f"Accuracy when agree:    {metrics['accuracy_when_agree']:.1%}")
    print(f"Accuracy when disagree: {metrics['accuracy_when_disagree']:.1%}")

TESTING DIFFERENT ENSEMBLE STRATEGIES

CONFIDENCE_WEIGHTED
------------------------------------------------------------
Hybrid Accuracy:        88.7%
Edit Distance Only:     88.7%
ML Model Only:          88.3%
Agreement Rate:         89.7%
Accuracy when agree:    98.1%
Accuracy when disagree: 6.5%

MAX_CONFIDENCE
------------------------------------------------------------
Hybrid Accuracy:        88.7%
Edit Distance Only:     88.7%
ML Model Only:          88.3%
Agreement Rate:         89.7%
Accuracy when agree:    98.1%
Accuracy when disagree: 6.5%

CONSERVATIVE
------------------------------------------------------------
Hybrid Accuracy:        88.7%
Edit Distance Only:     88.7%
ML Model Only:          88.3%
Agreement Rate:         89.7%
Accuracy when agree:    98.1%
Accuracy when disagree: 6.5%


## 8. Save Model for Production

Export trained model and lookup database for use in the backend API.

In [ ]:
import pickle
import os

# Create Models directory
os.makedirs('Models', exist_ok=True)

print("=" * 70)
print("SAVING MODELS (PHONETIC-AS-FEATURE VERSION)")
print("=" * 70)

# Verify ranker exists
print(f"✓ Ranker model exists: {ranker is not None}")
print(f"✓ Lookup database exists: {len(lookup_df)} rows")

# Save the model
model_path = 'Models/drug_ranker_model.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(ranker, f)
print(f"✓ Saved model to: {model_path}")
print(f"  Size: {os.path.getsize(model_path) / 1024:.2f} KB")

# Save lookup_df with NYSIIS codes
lookup_path = 'Models/lookup_df.pkl'

# Convert StringDtype to object for compatibility
lookup_df_save = lookup_df.copy()
for col in lookup_df_save.columns:
    if hasattr(lookup_df_save[col].dtype, 'storage'):
        lookup_df_save[col] = lookup_df_save[col].astype('object')
        print(f"  Converted {col} from StringDtype to object")

lookup_df_save.to_pickle(lookup_path)
print(f"✓ Saved lookup_df to: {lookup_path}")
print(f"  Size: {os.path.getsize(lookup_path) / 1024 / 1024:.2f} MB")

# Verify columns
print(f"\nLookup DataFrame columns:")
print(f"  {list(lookup_df_save.columns)}")

print("\n" + "=" * 70)
print("✅ PHONETIC-AS-FEATURE MODEL SAVED SUCCESSFULLY!")
print("=" * 70)

In [24]:
import os

print(f"Notebook is running from: {os.getcwd()}")
print(f"\nFiles in current directory:")
for file in os.listdir('.'):
    if file.endswith('.pkl'):
        print(f"  ✓ {file}")

Notebook is running from: C:\Users\User\OneDrive - University of Westminster\Medfind

Files in current directory:
  ✓ drug_ranker_model.pkl
  ✓ lookup_df.pkl


In [ ]:
import pickle
import os

print("="*70)
print("SAVING PICKLE FILES")
print("="*70)

# Check if variables exist
if 'ranker' not in globals():
    print("❌ ERROR: 'ranker' model doesn't exist!")
    print("   You need to run the training cells first")
else:
    print(f"✓ ranker model exists: {type(ranker)}")

if 'lookup_df' not in globals():
    print("❌ ERROR: 'lookup_df' doesn't exist!")
    print("   You need to run the data preparation cells first")
else:
    print(f"✓ lookup_df exists: {len(lookup_df)} rows")

# Only proceed if both exist
if 'ranker' in globals() and 'lookup_df' in globals():
    
    # Create directories
    backend_dir = r'C:\Users\User\OneDrive - University of Westminster\Medfind\Code\Backend\Models'
    os.makedirs(backend_dir, exist_ok=True)
    
    # Save model
    model_path = os.path.join(backend_dir, 'drug_ranker_model.pkl')
    with open(model_path, 'wb') as f:
        pickle.dump(ranker, f)
    
    model_size = os.path.getsize(model_path) / 1024 / 1024
    print(f"\n✓ Saved model to: {model_path}")
    print(f"  Size: {model_size:.2f} MB")
    
    # Save lookup_df
    lookup_path = os.path.join(backend_dir, 'lookup_df.pkl')
    lookup_df.to_pickle(lookup_path)
    
    lookup_size = os.path.getsize(lookup_path) / 1024 / 1024
    print(f"\n✓ Saved lookup_df to: {lookup_path}")
    print(f"  Size: {lookup_size:.2f} MB")
    
    print("\n" + "="*70)
    print("SUCCESS! Files saved to Backend/Models/")
    print("="*70)
  

else:
    print("\nCannot save, missing required variables")
    print("   Re-run the notebook from the beginning")

In [ ]:
#verify
print("\n VERIFICATION CHECKLIST:")
print(f"✓ Model trained: {ranker is not None}")
print(f"✓ Feature count: {ranker.num_feature()} (should be 12)")
print(f"✓ Lookup DB size: {len(lookup_df)} rows")
print(f"✓ Has NYSIIS codes: {'nysiis' in lookup_df.columns}")
print(f"✓ Has soundex codes: {'soundex' in lookup_df.columns}")
print(f"✓ Has metaphone codes: {'metaphone' in lookup_df.columns}")

# Test the dangerous query
print("\n TESTING DANGEROUS QUERY:")
test_result = rank_candidates("beprasone", lookup_df, ranker, top_k=5)
print(f"Top result for 'beprasone': {test_result.iloc[0]['canonical']}")
print(f"Should NOT be 'bevurogant' anymore!")

In [ ]:
# ============================================================
# DIAGNOSTIC: Check Saved Model
# ============================================================
# Run this cell in your notebook to verify the model

import pickle
import os

model_path = r'C:\Users\User\OneDrive - University of Westminster\Medfind\Code\Backend\Models\drug_ranker_model.pkl'

print("="*70)
print("CHECKING SAVED MODEL FILE")
print("="*70)

# Check file exists and size
if os.path.exists(model_path):
    size_mb = os.path.getsize(model_path) / 1024 / 1024
    print(f"\n✓ File exists: {model_path}")
    print(f"  Size: {size_mb:.2f} MB")
    
    # Load and inspect
    with open(model_path, 'rb') as f:
        saved_model = pickle.load(f)
    
    print(f"\n📊 Model Information:")
    print(f"  Type: {type(saved_model)}")
    print(f"  Features: {saved_model.num_feature()}")
    print(f"  Feature names: {saved_model.feature_name()}")
    
    # Check if it's the same as current ranker
    print(f"\n🔍 Comparison with Current Ranker:")
    print(f"  Current ranker features: {ranker.num_feature()}")
    print(f"  Current ranker names: {ranker.feature_name()}")
    
    if saved_model.feature_name() == ranker.feature_name():
        print(f"\n✓ Feature names MATCH!")
    else:
        print(f"\n❌ Feature names DON'T MATCH!")
        print(f"   Saved: {saved_model.feature_name()}")
        print(f"   Current: {ranker.feature_name()}")
    
    # Check model parameters
    print(f"\n📈 Model Parameters:")
    print(f"  Num trees (saved): {saved_model.num_trees()}")
    print(f"  Num trees (current): {ranker.num_trees()}")
    
    if saved_model.num_trees() == ranker.num_trees():
        print(f"\n✓ SAVED MODEL IS CURRENT!")
    else:
        print(f"\n⚠️  WARNING: Saved model has different number of trees!")
        print(f"   You need to SAVE AGAIN!")
        
else:
    print(f"\n❌ File not found: {model_path}")

print("\n" + "="*70)


# ============================================================
# IF MODEL IS WRONG, SAVE IT AGAIN
# ============================================================

print("\n" + "="*70)
print("RE-SAVING MODEL (FORCE UPDATE)")
print("="*70)

# Save current ranker
with open(model_path, 'wb') as f:
    pickle.dump(ranker, f)

size_mb = os.path.getsize(model_path) / 1024 / 1024
print(f"\n✓ Model saved to: {model_path}")
print(f"  New size: {size_mb:.2f} MB")

# Verify it saved correctly
with open(model_path, 'rb') as f:
    verify_model = pickle.load(f)

print(f"\n✓ Verification:")
print(f"  Features: {verify_model.num_feature()}")
print(f"  Feature names: {verify_model.feature_name()[:3]}...")
print(f"  Num trees: {verify_model.num_trees()}")

print("\n" + "="*70)
print("✅ MODEL FILE UPDATED!")
print("="*70)
print("\nNow restart your backend:")
print("  1. Stop backend (Ctrl+C)")
print("  2. Run: python medfind_backend.py")
print("  3. Test search for 'amoxcilin'")